# InstructLab Knowledge Q&A Generation — Tutorial

**Goal:** Automatically generate high-quality Q&A training data for [InstructLab](https://instructlab.ai/) knowledge contributions from documents.

## The Problem

InstructLab requires `qna.yaml` files with diverse, grounded question-answer pairs for knowledge contributions. Manually writing these is tedious and doesn't scale. Domain experts have knowledge but shouldn't need to understand YAML schemas.

## The Solution

This pipeline takes document chunks + taxonomy metadata and automatically:

1. **Generates diverse questions** across 5 categories (definitional, procedural, troubleshooting, comparative, best-practice)
2. **Produces grounded answers** using only information from the source document
3. **Evaluates faithfulness** via LLM-as-judge to filter hallucinated content
4. **Formats output** into valid InstructLab `qna.yaml` + `attribution.txt` files

```
  ┌───────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
  │ Generate   │   │ Generate │   │ Evaluate │   │ Format   │
  │ Questions  │──▶│ Answers  │──▶│Faithful- │──▶│ qna.yaml │
  │ (5/chunk)  │   │(grounded)│   │  ness    │   │          │
  └───────────┘   └──────────┘   └──────────┘   └──────────┘
```

---
## 0. Setup

Install dependencies and configure your API key.

In [ ]:
import os
import sys
import textwrap
from pathlib import Path

import pandas as pd
import nest_asyncio

nest_asyncio.apply()

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Configure your LLM — any OpenAI-compatible API works
API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...")
MODEL = os.environ.get("SDG_MODEL", "openai/gpt-5-mini")

print(f"Model: {MODEL}")

---
## 1. Prepare Input Data

The pipeline expects a DataFrame with three columns:

| Column | Description |
|--------|-------------|
| `document_text` | Pre-chunked document text (300-500 words) |
| `taxonomy_path` | InstructLab taxonomy placement |
| `domain` | Knowledge domain name |

Here we use a sample document about photosynthesis to demonstrate.

In [ ]:
# Sample document chunks — in production, you'd load and chunk real documents
documents = [
    {
        "document_text": (
            "Photosynthesis is the biological process by which green plants, algae, "
            "and certain bacteria convert light energy, usually from the sun, into "
            "chemical energy stored in glucose. This process occurs primarily in the "
            "chloroplasts of plant cells, specifically within structures called thylakoids "
            "that contain the green pigment chlorophyll. Photosynthesis consists of two "
            "main stages: the light-dependent reactions and the Calvin cycle (light-independent "
            "reactions). During the light-dependent reactions, chlorophyll absorbs sunlight "
            "and uses this energy to split water molecules (H2O) into oxygen, protons, and "
            "electrons. The oxygen is released as a byproduct — this is the source of most "
            "atmospheric oxygen on Earth. The energy captured is stored in ATP and NADPH. "
            "In the Calvin cycle, the enzyme RuBisCO fixes carbon dioxide (CO2) from the "
            "atmosphere, and using the ATP and NADPH from the light reactions, converts it "
            "into glucose (C6H12O6). This glucose serves as the primary energy source for "
            "the plant and forms the base of nearly all food chains on Earth. Factors "
            "affecting the rate of photosynthesis include light intensity, carbon dioxide "
            "concentration, temperature, and water availability. C4 and CAM plants have "
            "evolved alternative carbon fixation pathways to minimize photorespiration in "
            "hot, dry environments."
        ),
        "taxonomy_path": "knowledge/science/biology/photosynthesis",
        "domain": "biology",
    },
    {
        "document_text": (
            "Sourdough bread is made through a natural fermentation process using a "
            "sourdough starter — a mixture of flour and water that captures wild yeast "
            "and lactic acid bacteria from the environment. Unlike commercial bread that "
            "uses packaged yeast (Saccharomyces cerevisiae), sourdough relies on a "
            "symbiotic culture of wild yeast species (often Kazachstania humilis) and "
            "Lactobacillus bacteria. The fermentation process typically takes 12-24 hours, "
            "much longer than conventional bread (2-4 hours). During fermentation, the "
            "bacteria produce lactic and acetic acids, giving sourdough its characteristic "
            "tangy flavor and lowering the pH to 3.5-4.5. This acidic environment also "
            "breaks down phytic acid, making minerals more bioavailable, and partially "
            "degrades gluten proteins, which some people find easier to digest. Common "
            "troubleshooting issues include: a starter that won't rise (often due to "
            "cold temperatures below 70°F/21°C), overly sour flavor (too much acetic "
            "acid from cold fermentation), dense crumb structure (insufficient gluten "
            "development or underproofing), and poor oven spring (inadequate steam or "
            "scoring). Maintaining a healthy starter requires regular feeding — typically "
            "discarding half and adding equal parts flour and water every 12-24 hours at "
            "room temperature, or weekly if refrigerated."
        ),
        "taxonomy_path": "knowledge/food/baking/sourdough",
        "domain": "culinary arts",
    },
]

input_df = pd.DataFrame(documents)
print(f"Input: {len(input_df)} document chunks")
print(f"Taxonomy paths: {input_df['taxonomy_path'].tolist()}")
print(f"\nFirst chunk preview ({len(input_df.iloc[0]['document_text'].split())} words):")
print(textwrap.fill(input_df.iloc[0]["document_text"][:200] + "...", width=80))

---
## 2. Load the Flow

The InstructLab Q&A flow is registered in SDG Hub's flow registry. It uses 16 blocks across 4 stages.

In [ ]:
from sdg_hub import Flow, FlowRegistry

# Discover all available flows
FlowRegistry.discover_flows()

# Load our flow by ID
flow_id = "bright-coral-421"
flow_path = FlowRegistry.get_flow_path(flow_id)
flow = Flow.from_yaml(flow_path)

print(f"Flow: {flow.metadata.name}")
print(f"Version: {flow.metadata.version}")
print(f"Tags: {flow.metadata.tags}")
print(f"Blocks: {len(flow.blocks)}")
print(f"\nRequired input columns: {flow.metadata.dataset_requirements.required_columns}")

In [ ]:
flow.print_info()

---
## 3. Configure the Model

The flow uses LLM calls for question generation, answer generation, and faithfulness evaluation. Configure your model and API key here.

In [ ]:
flow.set_model_config(
    model=MODEL,
    api_key=API_KEY,
)
print(f"Model configured: {MODEL}")

---
## 4. Run the Pipeline

The pipeline processes each document chunk through 4 stages:

1. **Generate Questions** — 5 diverse questions per chunk
2. **Generate Answers** — grounded in source document
3. **Evaluate Faithfulness** — LLM-as-judge filters hallucinated content
4. **Format qna.yaml** — groups by taxonomy path, produces valid InstructLab files

With 2 input chunks, expect ~10 candidate Q&A pairs → filtered to faithful ones → grouped into 2 `qna.yaml` files.

In [ ]:
print(f"Running pipeline on {len(input_df)} document chunks...")
print(f"Expected: ~{len(input_df) * 5} Q&A candidates before filtering\n")

result = flow.generate(input_df)

In [ ]:
if hasattr(result, "to_pandas"):
    result_df = result.to_pandas()
else:
    result_df = result

print(f"Pipeline complete!")
print(f"Generated {len(result_df)} qna.yaml document(s)")
print(f"Columns: {list(result_df.columns)}")
display(result_df[["taxonomy_path", "num_examples"]])

---
## 5. Inspect the Output

Each row contains a complete `qna.yaml` document and `attribution.txt` ready for an InstructLab PR.

In [ ]:
# Show the first qna.yaml
for i, row in result_df.iterrows():
    print(f"{'=' * 70}")
    print(f"Taxonomy: {row['taxonomy_path']}")
    print(f"Examples: {row['num_examples']}")
    print(f"{'=' * 70}")
    print()
    print("--- qna.yaml ---")
    print(row["qna_yaml"])
    print("--- attribution.txt ---")
    print(row["attribution_txt"])
    print()

---
## 6. Export for InstructLab

Save the generated files to disk in the directory structure InstructLab expects.

In [ ]:
output_dir = NOTEBOOK_DIR / "output"

for _, row in result_df.iterrows():
    # Create directory matching taxonomy path
    tax_dir = output_dir / row["taxonomy_path"]
    tax_dir.mkdir(parents=True, exist_ok=True)

    # Write qna.yaml
    qna_path = tax_dir / "qna.yaml"
    qna_path.write_text(row["qna_yaml"])

    # Write attribution.txt
    attr_path = tax_dir / "attribution.txt"
    attr_path.write_text(row["attribution_txt"])

    print(f"Written: {qna_path}")
    print(f"Written: {attr_path}")

print(f"\nAll files exported to {output_dir}/")
print("These can be submitted as an InstructLab taxonomy PR.")

---
## 7. Scaling Up

### More documents

Add more rows to the input DataFrame — each document chunk is processed independently:

```python
# Load and chunk your documents
chunks = chunk_my_documents("path/to/docs/", chunk_size=400)
input_df = pd.DataFrame(chunks)
result = flow.generate(input_df)
```

### Different models

The flow works with any OpenAI-compatible API. For Granite models:

```python
flow.set_model_config(
    model="ibm/granite-3.3-8b-instruct",
    api_key=os.environ["GRANITE_API_KEY"],
    api_base="https://your-granite-endpoint/v1",
)
```

### Custom formatting

Override the `InstructLabFormatterBlock` parameters at runtime:

```python
result = flow.generate(
    input_df,
    runtime_params={
        "format_qna_yaml": {
            "created_by": "your-github-username",
            "min_examples": 3,  # Lower threshold for small documents
        },
    },
)
```

---

## What we built

This tutorial walked through the **InstructLab Knowledge Q&A Generation** pipeline — an automated way to produce training data for InstructLab knowledge contributions.

### The pipeline

1. **Question Generation** — An LLM generates 5 diverse questions per document chunk, covering definitions, processes, troubleshooting, comparisons, and best practices
2. **Answer Generation** — A second LLM call produces detailed answers grounded strictly in the source document
3. **Faithfulness Evaluation** — An LLM-as-judge scores each Q&A pair and filters out any with hallucinated content
4. **YAML Formatting** — The `InstructLabFormatterBlock` groups pairs by taxonomy path and outputs valid `qna.yaml` + `attribution.txt`

### What makes this useful

- **End-to-end** — Goes from raw documents to submission-ready InstructLab files
- **Quality-gated** — Every Q&A pair is evaluated for faithfulness before inclusion
- **Model-agnostic** — Works with any OpenAI-compatible LLM (GPT, Granite, Llama, etc.)
- **Composable** — Built on SDG Hub blocks, so you can customize or extend any stage